<a href="https://colab.research.google.com/github/e2458115-hash/fes-traffic-game/blob/main/colab-engine/route_engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q fastapi uvicorn networkx nest-asyncio

In [3]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null

import time
import subprocess
import re
import nest_asyncio
import uvicorn
import networkx as nx
from fastapi import FastAPI
from pydantic import BaseModel

nest_asyncio.apply()

app = FastAPI()
G = nx.DiGraph()

NODES = {
    "GATE_MAIN": {"x": 80,  "y": 250, "type": "gate"},
    "GATE_WEST": {"x": 80,  "y": 80,  "type": "gate"},
    "JCT_1":     {"x": 260, "y": 165, "type": "junction"},
    "JCT_2":     {"x": 440, "y": 165, "type": "junction"},
    "PARK_A":    {"x": 620, "y": 80,  "type": "parking"},
    "PARK_B":    {"x": 620, "y": 250, "type": "parking"},
}

for node, attr in NODES.items():
    G.add_node(node, **attr)

EDGES = [
    ("GATE_MAIN", "JCT_1", 10),
    ("GATE_WEST", "JCT_1", 12),
    ("JCT_1", "JCT_2", 15),
    ("JCT_1", "PARK_B", 25),
    ("JCT_2", "PARK_A", 10),
    ("JCT_2", "PARK_B", 10),
]

for u, v, w in EDGES:
    G.add_edge(u, v, weight=w)

class RouteRequest(BaseModel):
    start_node: str
    target_parking: str
    congestion_jct1: int = 0

@app.get("/map")
def get_map():
    return {
        "nodes": NODES,
        "edges": [{"from": u, "to": v, "weight": d["weight"]} for u, v, d in G.edges(data=True)]
    }

@app.post("/route")
def calculate_route(req: RouteRequest):
    start_time = time.perf_counter()
    temp_G = G.copy()
    if temp_G.has_edge("JCT_1", "JCT_2"):
        temp_G["JCT_1"]["JCT_2"]["weight"] += req.congestion_jct1

    try:
        path = nx.shortest_path(temp_G, source=req.start_node, target=req.target_parking, weight="weight")
        cost = nx.shortest_path_length(temp_G, source=req.start_node, target=req.target_parking, weight="weight")
        centrality = nx.betweenness_centrality(temp_G, weight="weight")
        elapsed_ms = round((time.perf_counter() - start_time) * 1000, 3)

        return {
            "status": "ok",
            "path": path,
            "coords": [NODES[n] for n in path],
            "cost": cost,
            "bottleneck_node": max(centrality, key=centrality.get),
            "py_exec_ms": elapsed_ms
        }
    except nx.NetworkXNoPath:
        return {"status": "error", "message": "No path found"}

tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
for line in tunnel_proc.stdout:
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        print("==========================================")
        print(f"🚀 Colab接続用URL: {public_url}")
        print("==========================================")
        break

config = uvicorn.Config(app, host="127.0.0.1", port=8000)
server = uvicorn.Server(config)
await server.serve()

🚀 Colab接続用URL: https://myspace-tracy-joel-craps.trycloudflare.com


INFO:     Started server process [725]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     150.37.251.34:0 - "GET /map HTTP/1.1" 200 OK
INFO:     150.37.251.34:0 - "POST /route HTTP/1.1" 200 OK
INFO:     150.37.251.34:0 - "POST /route HTTP/1.1" 200 OK
INFO:     150.37.251.34:0 - "POST /route HTTP/1.1" 200 OK
INFO:     150.37.251.34:0 - "POST /route HTTP/1.1" 200 OK
INFO:     150.37.251.34:0 - "POST /route HTTP/1.1" 200 OK
INFO:     150.37.251.34:0 - "POST /route HTTP/1.1" 200 OK
INFO:     150.37.251.34:0 - "POST /route HTTP/1.1" 200 OK
INFO:     150.37.251.34:0 - "POST /route HTTP/1.1" 200 OK
INFO:     150.37.251.34:0 - "POST /route HTTP/1.1" 200 OK
INFO:     150.37.251.34:0 - "POST /route HTTP/1.1" 200 OK
INFO:     150.37.251.34:0 - "GET /map HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [725]
